# Learning to See in the Dark -- Kaggle GPU T4x2

**Paper**: *Learning to See in the Dark*, Chen Chen, Qifeng Chen, Jia Xu, Vladlen Koltun -- CVPR 2018

## Key Contributions
1. **SID Dataset** -- 5,094 raw short-exposure / long-exposure image pairs (Sony a7SII + Fujifilm X-T2).
2. **End-to-End ISP** -- A fully-convolutional network that operates *directly on 14-bit raw sensor data*.
3. **U-Net + PixelShuffle** -- Input: 4-channel packed Bayer (H/2 x W/2). Output: 12ch -> PixelShuffle(2) -> full-resolution RGB.
4. **Training** -- L1 loss, Adam, 4,000 epochs, 512x512 random crops.

## Notebook Overview
| # | Section | Description |
|---|---------|-------------|
| 1 | Environment Setup | GPU check, multi-GPU detection, system info |
| 2 | Configuration | All hyperparameters and paths |
| 3 | Dataset Download | Download Sony 2025 subset, selective 20% extraction |
| 4 | File-List Parsing | Parse SID list files, apply 20% scene filter |
| 5 | **EDA** | Dataset stats, exposure distributions, sample images, channel stats |
| 6 | Preprocessing | Bayer packing, black-level correction, amplification |
| 7 | Dataset & DataLoader | SIDDataset class, loaders, sanity checks |
| 8 | Model | U-Net with PixelShuffle output |
| 9 | Training | AMP mixed-precision, multi-GPU DataParallel |
| 10 | Training Curves | Loss / PSNR / SSIM plots |
| 11 | Evaluation | Test metrics, per-sample analysis, error maps |
| 12 | Performance Summary | Comparison with paper baselines |
| 13 | Inference | Single-image inference demo |

> **Resource optimisations** -- Images scaled to 25% (`SCALE_FACTOR=0.25`), patch size 256, only **20% of scenes** are used. Both T4 GPUs are utilised via `nn.DataParallel`.

---
## Section 1 -- Environment Setup

In [ ]:
# Install required packages (Kaggle pre-installs most; rawpy may need explicit install)
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["rawpy", "imageio", "scikit-image", "tqdm"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

print("All packages available.")

In [ ]:
# -- Core imports --
import os
import re
import sys
import glob
import math
import time
import json
import random
import zipfile
import urllib.request
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import rawpy
import imageio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.transform import resize as sk_resize

# Plotting style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("All imports successful.")

In [ ]:
# -- GPU / hardware detection --
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"GPU count       : {n_gpus}")

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB")

if n_gpus == 0:
    warnings.warn("No GPU found -- enable: Settings -> Accelerator -> GPU T4 x2")

try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                ram_gb = int(line.split()[1]) / 1e6
                print(f"System RAM      : {ram_gb:.1f} GB")
                break
except Exception:
    pass

import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk (working)  : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

---
## Section 2 -- Configuration

In [ ]:
# =========================================================================
#  CONFIGURATION -- adjust these to trade speed / quality / memory usage
# =========================================================================

# -- Paths --
# Input dataset mounted from Kaggle (read-only)
DATASET_DIR  = "/kaggle/input/datasets/marcorosato/sid-dataset"
# Writable working directory (checkpoints, results, downloaded list files)
WORK_DIR     = Path("/kaggle/working")
CKPT_DIR     = Path("/kaggle/working/checkpoints")
RESULT_DIR   = Path("/kaggle/working/results")
LISTS_BASE   = "https://raw.githubusercontent.com/cchen156/Learning-to-See-in-the-Dark/master"

WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# -- Dataset sampling --
# Only use this fraction of unique scenes to save RAM and VRAM.
SAMPLE_FRAC  = 0.20   # 20% of all scenes

# -- Image scaling --
# SCALE_FACTOR is applied to the raw image before Bayer packing.
# 0.25 = quarter resolution -> ~16x smaller tensors vs. full-res.
SCALE_FACTOR = 0.25

# PATCH_SIZE -- crop size (raw pixels before packing). Must be even.
PATCH_SIZE   = 256

# -- Training hyperparameters --
BATCH_SIZE   = 8      # increase if VRAM allows (T4 x2 ~ 30 GB total)
NUM_EPOCHS   = 500    # reduce to 100 for a quick demo
LR_INITIAL   = 1e-4
LR_DECAY_EP  = 250    # halve LR at this epoch
LR_DECAYED   = 1e-5
NUM_WORKERS  = 4

# -- Misc --
SEED         = 42
SAVE_EVERY   = 50     # save periodic checkpoint every N epochs
VAL_EVERY    = 25     # run validation every N epochs

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration summary:")
print(f"  DATASET_DIR  = {DATASET_DIR}")
print(f"  SAMPLE_FRAC  = {SAMPLE_FRAC} ({SAMPLE_FRAC*100:.0f}% of scenes)")
print(f"  SCALE_FACTOR = {SCALE_FACTOR}  (images scaled to {SCALE_FACTOR*100:.0f}%)")
print(f"  PATCH_SIZE   = {PATCH_SIZE}")
print(f"  BATCH_SIZE   = {BATCH_SIZE}")
print(f"  NUM_EPOCHS   = {NUM_EPOCHS}")
print(f"  NUM_WORKERS  = {NUM_WORKERS}")


---
## Section 3 -- 20% Scene Sampling from Kaggle Input Dataset

The dataset is pre-mounted (read-only) at `/kaggle/input/datasets/marcorosato/sid-dataset`.
No download or extraction is required.

We sample **20% of unique scene IDs** from the available `.ARW` files so that
training, validation, and test sets remain consistent and reproducible:

1. Scan `Sony/short/` and `Sony/long/` for all `.ARW` files.
2. Collect the set of 5-digit scene IDs (e.g. `00001`).
3. Randomly sample 20% of those scene IDs (fixed seed for reproducibility).
4. All subsequent parsing steps filter pairs to only these scene IDs.


In [ ]:
# =========================================================================
#  SECTION 3 -- 20% Selective Scene Sampling from Kaggle Input Dataset
# =========================================================================

from tqdm.auto import tqdm

SONY_SHORT = Path(DATASET_DIR) / "Sony" / "short"
SONY_LONG  = Path(DATASET_DIR) / "Sony" / "long"

# Verify dataset directories exist
if not SONY_SHORT.exists() or not SONY_LONG.exists():
    raise FileNotFoundError(
        f"Expected Sony dataset directories not found.\n"
        f"  short: {SONY_SHORT}\n"
        f"  long : {SONY_LONG}\n"
        "Ensure the dataset 'marcorosato/sid-dataset' is added to this notebook."
    )

# Scan all ARW/RAF files to collect scene IDs
print("Scanning dataset for scene IDs...")
arw_files = (
    list(SONY_SHORT.glob("*.ARW")) + list(SONY_SHORT.glob("*.RAF")) +
    list(SONY_LONG.glob("*.ARW"))  + list(SONY_LONG.glob("*.RAF"))
)

scene_id_set = set()
for f in arw_files:
    m = re.match(r"(\d{5})_", f.name)
    if m:
        scene_id_set.add(m.group(1))

scene_ids = sorted(scene_id_set)
n_sample  = max(1, int(len(scene_ids) * SAMPLE_FRAC))
random.seed(SEED)
sampled_scenes = set(random.sample(scene_ids, n_sample))

print(f"Total unique scenes in dataset : {len(scene_ids)}")
print(f"Sampled scenes (20%)           : {len(sampled_scenes)}")

# Persist the scene list to the writable working directory
SC_FILE = WORK_DIR / "sampled_scenes.txt"
with open(SC_FILE, "w") as f:
    f.write("\n".join(sorted(sampled_scenes)))
print(f"Saved sampled scene list -> {SC_FILE}")


---
## Section 4 -- File-List Parsing

In [ ]:
# -- Download official train/val/test split lists --
# List files are saved to WORK_DIR (/kaggle/working) because the input
# dataset directory is read-only.
LISTS_DIR = WORK_DIR

for split in ["train", "val", "test"]:
    fname = f"Sony_{split}_list.txt"
    dest  = LISTS_DIR / fname
    if not dest.exists():
        url = f"{LISTS_BASE}/{fname}"
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, str(dest))
        print(f"Saved -> {dest}")
    else:
        print(f"{fname} already present.")


In [ ]:
# -- File-list parsing helpers --

def extract_exposure(filename: str) -> float:
    """Parse exposure time (seconds) from SID filename.
    E.g. 00001_00_0.1s.ARW -> 0.1
    """
    stem = Path(filename).stem
    t = stem.split("_")[-1]
    return float(t.rstrip("sS"))


def extract_scene_id(filename: str) -> str:
    base = os.path.basename(filename)
    m = re.match(r"(\d{5})_", base)
    return m.group(1) if m else "unknown"


def parse_sony_list(txt_path, sony_root, allowed_scenes=None):
    """Parse a SID Sony_*_list.txt.
    Returns list of dicts: input_path, gt_path, amp_ratio, scene_id, t_in, t_gt.
    """
    pairs = []
    sony_root = Path(sony_root)
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts   = line.split()
            in_rel  = parts[0].lstrip("./")
            gt_rel  = parts[1].lstrip("./")
            in_path = sony_root / in_rel
            gt_path = sony_root / gt_rel
            if not in_path.exists() or not gt_path.exists():
                continue
            scene_id = extract_scene_id(str(in_path))
            if allowed_scenes is not None and scene_id not in allowed_scenes:
                continue
            t_in = extract_exposure(str(in_path))
            t_gt = extract_exposure(str(gt_path))
            amp  = t_gt / t_in if t_in > 0 else 300.0
            pairs.append({
                "input_path": str(in_path),
                "gt_path":    str(gt_path),
                "amp_ratio":  amp,
                "scene_id":   scene_id,
                "t_in":       t_in,
                "t_gt":       t_gt,
            })
    return pairs


# SONY_DIR  -- input (read-only), contains Sony/short/ and Sony/long/
# LISTS_DIR -- writable working dir, contains the downloaded *_list.txt files
SONY_DIR    = Path(DATASET_DIR)
train_pairs = parse_sony_list(LISTS_DIR / "Sony_train_list.txt", SONY_DIR, sampled_scenes)
val_pairs   = parse_sony_list(LISTS_DIR / "Sony_val_list.txt",   SONY_DIR, sampled_scenes)
test_pairs  = parse_sony_list(LISTS_DIR / "Sony_test_list.txt",  SONY_DIR, sampled_scenes)

print(f"Train: {len(train_pairs):4d} pairs  ({len(set(p["scene_id"] for p in train_pairs))} scenes)")
print(f"Val  : {len(val_pairs):4d} pairs  ({len(set(p["scene_id"] for p in val_pairs))} scenes)")
print(f"Test : {len(test_pairs):4d} pairs  ({len(set(p["scene_id"] for p in test_pairs))} scenes)")

---
## Section 5 -- Exploratory Data Analysis (EDA)

We examine the 20% subset before training to understand:
- **Split sizes** and scene coverage
- **Exposure time** distributions (short and long)
- **Amplification ratio** distribution
- **Visual inspection** of raw inputs and GT images
- **Pixel intensity** statistics per Bayer channel

In [ ]:
# -- 5.1  Dataset overview table --
all_pairs = train_pairs + val_pairs + test_pairs
splits  = ["Train", "Val", "Test"]
counts  = [len(train_pairs), len(val_pairs), len(test_pairs)]
scenes  = [len(set(p["scene_id"] for p in sp))
           for sp in [train_pairs, val_pairs, test_pairs]]

print("=" * 50)
print("  DATASET OVERVIEW (20% subset)")
print("=" * 50)
print(f"  {'Split':<10}  {'Pairs':<10}  {'Unique Scenes'}")
print("-" * 50)
for s, c, sc in zip(splits, counts, scenes):
    print(f"  {s:<10}  {c:<10}  {sc}")
print("-" * 50)
print(f"  {'Total':<10}  {sum(counts):<10}  "
      f"{len(set(p['scene_id'] for p in all_pairs))}")
print("=" * 50)

amps  = [p["amp_ratio"] for p in all_pairs]
t_ins = [p["t_in"] for p in all_pairs]
t_gts = [p["t_gt"] for p in all_pairs]

print(f"\nAmplification: min={min(amps):.0f}x  max={max(amps):.0f}x  "
      f"mean={np.mean(amps):.1f}x  median={np.median(amps):.0f}x")
print(f"Short exp (s): min={min(t_ins):.3f}  max={max(t_ins):.3f}  mean={np.mean(t_ins):.3f}")
print(f"Long  exp (s): min={min(t_gts):.1f}  max={max(t_gts):.1f}  mean={np.mean(t_gts):.1f}")

In [ ]:
# -- 5.2  Split distribution & amplification histogram --
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Dataset Overview -- 20% Subset", fontsize=14, fontweight="bold")

# Pie: train / val / test
axes[0].pie(counts, labels=splits, autopct="%1.1f%%",
            colors=["#4C72B0", "#DD8452", "#55A868"],
            startangle=140, pctdistance=0.8)
axes[0].set_title("Split Distribution")

# Short exposure histogram
t_split = {s: [p["t_in"] for p in sp]
           for s, sp in zip(splits, [train_pairs, val_pairs, test_pairs])}
bins = np.logspace(np.log10(max(1e-4, min(t_ins))),
                   np.log10(max(t_ins) + 0.01), 30)
colors_s = ["#4C72B0", "#DD8452", "#55A868"]
for (s, vals), col in zip(t_split.items(), colors_s):
    axes[1].hist(vals, bins=bins, alpha=0.7, label=s, color=col)
axes[1].set_xscale("log")
axes[1].set_xlabel("Short Exposure Time (s)")
axes[1].set_ylabel("Count")
axes[1].set_title("Short Exposure Distribution")
axes[1].legend()

# Amplification histogram
axes[2].hist(amps, bins=30, color="#8172B2", edgecolor="white")
axes[2].axvline(np.mean(amps), color="red",    linestyle="--",
               label=f"Mean={np.mean(amps):.0f}x")
axes[2].axvline(np.median(amps), color="orange", linestyle="--",
               label=f"Median={np.median(amps):.0f}x")
axes[2].set_xlabel("Amplification Ratio")
axes[2].set_ylabel("Count")
axes[2].set_title("Amplification Ratio Distribution")
axes[2].legend()

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_overview.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5.3  Pairs per amplification bucket --
amp_buckets = defaultdict(int)
for p in train_pairs:
    bucket = int(round(p["amp_ratio"] / 50.0)) * 50
    amp_buckets[bucket] += 1

fig, ax = plt.subplots(figsize=(10, 4))
xs = sorted(amp_buckets.keys())
ys = [amp_buckets[x] for x in xs]
bars = ax.bar([str(x) + "x" for x in xs], ys, color="#4C72B0", edgecolor="white")
ax.bar_label(bars, padding=2)
ax.set_xlabel("Amplification Ratio Bucket")
ax.set_ylabel("Number of Training Pairs")
ax.set_title("Training Pairs per Amplification Ratio Bucket")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_amp_buckets.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5.4  Sample raw and GT images --
N_SHOW = min(3, len(train_pairs))
fig, axes = plt.subplots(N_SHOW, 3, figsize=(15, 5 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle("Sample Image Pairs  |  Short Exposure  |  Bayer Channels  |  GT",
             fontsize=13, fontweight="bold")

for row, idx in enumerate(random.sample(range(len(train_pairs)), N_SHOW)):
    pair = train_pairs[idx]
    amp  = pair["amp_ratio"]
    with rawpy.imread(pair["input_path"]) as raw:
        bayer = raw.raw_image_visible.copy().astype(np.float32)
        blk   = int(raw.black_level_per_channel[0])
        wht   = int(raw.white_level)

    bayer_norm = np.clip((bayer - blk) / (wht - blk), 0, 1)
    H, W = bayer_norm.shape
    H -= H % 2; W -= W % 2
    b = bayer_norm[:H, :W]
    R  = b[0::2, 0::2]
    Gr = b[0::2, 1::2]
    B  = b[1::2, 1::2]
    Gb = b[1::2, 0::2]

    pseudo_rgb = np.stack([R * amp, (Gr + Gb) / 2 * amp, B * amp], axis=-1)
    pseudo_rgb = np.clip(pseudo_rgb, 0, 1)
    ch_map = np.stack([R, (Gr + Gb) / 2, B], axis=-1)

    with rawpy.imread(pair["gt_path"]) as raw:
        gt_rgb = raw.postprocess(use_camera_wb=True, half_size=True,
                                 no_auto_bright=True, output_bps=8)
    gt_rgb = gt_rgb.astype(np.float32) / 255.0

    axes[row, 0].imshow(pseudo_rgb)
    axes[row, 0].set_title(f"Raw input (amp x{amp:.0f})  scene {pair["scene_id"]}",
                           fontsize=9)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(ch_map)
    axes[row, 1].set_title("Bayer channels (R / G / B)", fontsize=9)
    axes[row, 1].axis("off")
    axes[row, 2].imshow(gt_rgb)
    axes[row, 2].set_title(f"Ground-truth (t={pair["t_gt"]}s)", fontsize=9)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_sample_images.png"), dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5.5  Pixel intensity & channel statistics --
N_STAT = min(10, len(train_pairs))
sample_stats = {"R": [], "Gr": [], "B": [], "Gb": []}

print(f"Computing channel statistics on {N_STAT} random training images...")
for pair in tqdm(random.sample(train_pairs, N_STAT)):
    with rawpy.imread(pair["input_path"]) as raw:
        bayer = raw.raw_image_visible.copy().astype(np.float32)
        blk   = int(raw.black_level_per_channel[0])
        wht   = int(raw.white_level)
    bayer_norm = np.clip((bayer - blk) / (wht - blk), 0, 1)
    H, W = bayer_norm.shape
    H -= H % 2; W -= W % 2
    b = bayer_norm[:H, :W]
    sample_stats["R"].append(b[0::2, 0::2].flatten())
    sample_stats["Gr"].append(b[0::2, 1::2].flatten())
    sample_stats["B"].append(b[1::2, 1::2].flatten())
    sample_stats["Gb"].append(b[1::2, 0::2].flatten())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Bayer Channel Pixel Intensity Distributions (no amplification)",
             fontsize=13, fontweight="bold")
ch_colors = {"R": "red", "Gr": "green", "B": "blue", "Gb": "darkgreen"}

for ax, (ch, vals_list) in zip(axes.flatten(), sample_stats.items()):
    vals = np.concatenate(vals_list)
    ax.hist(vals, bins=100, color=ch_colors[ch], alpha=0.75, density=True)
    ax.axvline(np.mean(vals), color="black", linestyle="--",
               label=f"mean={np.mean(vals):.3f}")
    ax.axvline(np.median(vals), color="gray", linestyle=":",
               label=f"median={np.median(vals):.3f}")
    ax.set_title(f"Channel {ch}")
    ax.set_xlabel("Normalised pixel value"); ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    print(f"  Channel {ch}: mean={np.mean(vals):.4f}  "
          f"std={np.std(vals):.4f}  p95={np.percentile(vals, 95):.4f}")

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_channel_stats.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5.6  Bayer channel correlation --
pair0 = train_pairs[0]
with rawpy.imread(pair0["input_path"]) as raw:
    bayer = raw.raw_image_visible.copy().astype(np.float32)
    blk   = int(raw.black_level_per_channel[0])
    wht   = int(raw.white_level)

bayer_norm = np.clip((bayer - blk) / (wht - blk), 0, 1)
H, W = bayer_norm.shape
H -= H % 2; W -= W % 2
b = bayer_norm[:H, :W]

R  = b[0::2, 0::2].flatten()
Gr = b[0::2, 1::2].flatten()
B  = b[1::2, 1::2].flatten()
Gb = b[1::2, 0::2].flatten()

idx_s = np.random.choice(len(R), size=min(5000, len(R)), replace=False)
data_mat = np.stack([R[idx_s], Gr[idx_s], B[idx_s], Gb[idx_s]])
corr = np.corrcoef(data_mat)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(["R", "Gr", "B", "Gb"])
ax.set_yticklabels(["R", "Gr", "B", "Gb"])
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center",
                fontsize=11,
                color="black" if abs(corr[i,j]) < 0.7 else "white")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Bayer Channel Correlation (1 image sample)")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_channel_corr.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 5.7  Noise level vs amplification ratio --
N_NOISE = min(15, len(train_pairs))
noise_levels = []
amp_vals     = []

print(f"Estimating noise on {N_NOISE} samples...")
for pair in tqdm(random.sample(train_pairs, N_NOISE)):
    with rawpy.imread(pair["input_path"]) as raw:
        bayer = raw.raw_image_visible.copy().astype(np.float32)
        blk   = int(raw.black_level_per_channel[0])
        wht   = int(raw.white_level)
    b_norm = np.clip((bayer - blk) / (wht - blk), 0, 1)
    noise_levels.append(float(np.std(b_norm)))
    amp_vals.append(pair["amp_ratio"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(amp_vals, noise_levels, alpha=0.7, s=60, c="#4C72B0")
ax.set_xlabel("Amplification Ratio")
ax.set_ylabel("Pixel Std-Dev (raw, proxy for noise)")
ax.set_title("Noise Level vs Amplification Ratio")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "eda_noise_vs_amp.png"), dpi=150, bbox_inches="tight")
plt.show()
print("EDA complete.")

---
## Section 6 -- Raw Image Preprocessing

### Bayer Packing
The Sony camera outputs a single-channel 14-bit RGGB Bayer image.
We pack it into **4 channels** and halve the spatial resolution:
```
channel 0 -> R   [rows 0::2, cols 0::2]
channel 1 -> Gr  [rows 0::2, cols 1::2]
channel 2 -> B   [rows 1::2, cols 1::2]
channel 3 -> Gb  [rows 1::2, cols 0::2]
```
An H x W raw -> (H/2) x (W/2) x 4 tensor after packing.

### Amplification
```
amp = t_long / t_short   (typically 100-300x)
```

In [ ]:
# -- Preprocessing functions --

def pack_raw_bayer(raw_image: np.ndarray,
                   black_level: int = 512,
                   white_level: int = 16383) -> np.ndarray:
    """Pack single-channel 14-bit Bayer (H x W) into 4-channel float32 (H/2 x W/2 x 4)."""
    im = raw_image.astype(np.float32)
    im = np.maximum(im - black_level, 0) / (white_level - black_level)
    H, W = im.shape
    H -= H % 2;  W -= W % 2
    im = im[:H, :W]
    return np.stack([
        im[0::2, 0::2],   # R
        im[0::2, 1::2],   # Gr
        im[1::2, 1::2],   # B
        im[1::2, 0::2],   # Gb
    ], axis=-1)


def load_input_raw(path: str, amp_ratio: float,
                   scale: float = 1.0) -> np.ndarray:
    """Load short-exposure ARW -> packed, amplified float32 (H', W', 4) in [0,1]."""
    with rawpy.imread(path) as raw:
        bayer = raw.raw_image_visible.copy()
        black = int(raw.black_level_per_channel[0])
        white = int(raw.white_level)

    if scale != 1.0:
        H, W  = bayer.shape
        new_H = max(2, int(round(H * scale)))
        new_W = max(2, int(round(W * scale)))
        new_H -= new_H % 2;  new_W -= new_W % 2
        step_H = max(1, H // new_H)
        step_W = max(1, W // new_W)
        bayer  = bayer[:new_H * step_H:step_H, :new_W * step_W:step_W]

    packed = pack_raw_bayer(bayer, black_level=black, white_level=white)
    return np.clip(packed * amp_ratio, 0.0, 1.0)


def load_gt_rgb(path: str, scale: float = 1.0) -> np.ndarray:
    """Load long-exposure ARW -> float32 sRGB (H, W, 3) in [0,1]."""
    with rawpy.imread(path) as raw:
        rgb = raw.postprocess(
            use_camera_wb=True, half_size=False,
            no_auto_bright=True, output_bps=16)
    gt = rgb.astype(np.float32) / 65535.0
    if scale != 1.0:
        H, W  = gt.shape[:2]
        new_H = max(1, int(round(H * scale)))
        new_W = max(1, int(round(W * scale)))
        gt    = sk_resize(gt, (new_H, new_W),
                          anti_aliasing=True,
                          preserve_range=True).astype(np.float32)
    return gt


print("Preprocessing utilities defined.")

In [ ]:
# -- Preprocessing visualisation --
pair  = train_pairs[0]
packed = load_input_raw(pair["input_path"], pair["amp_ratio"], scale=SCALE_FACTOR)
gt     = load_gt_rgb(pair["gt_path"], scale=SCALE_FACTOR)

print(f"Packed input shape : {packed.shape}   dtype={packed.dtype}")
print(f"GT image shape     : {gt.shape}   dtype={gt.dtype}")
print(f"Packed value range : [{packed.min():.3f}, {packed.max():.3f}]")
print(f"GT value range     : [{gt.min():.3f}, {gt.max():.3f}]")

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f"Preprocessing output at SCALE_FACTOR={SCALE_FACTOR}",
             fontsize=12, fontweight="bold")
ch_names = ["R (ch0)", "Gr (ch1)", "B (ch2)", "Gb (ch3)"]
for i in range(4):
    axes[i].imshow(packed[:, :, i], cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(ch_names[i], fontsize=9)
    axes[i].axis("off")
axes[4].imshow(gt)
axes[4].set_title("GT sRGB", fontsize=9)
axes[4].axis("off")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "preprocessing_vis.png"), dpi=120, bbox_inches="tight")
plt.show()

---
## Section 7 -- Dataset & DataLoader

In [ ]:
# -- SIDDataset --

class SIDDataset(Dataset):
    """Sony SID dataset.

    Training mode -> random crop + augment.
    Eval mode     -> full (scaled) image, no crop.
    """

    def __init__(self, pairs, patch_size=256, scale=1.0,
                 is_train=True, cache_gt=False):
        self.pairs      = pairs
        self.patch_size = patch_size
        self.scale      = scale
        self.is_train   = is_train
        self.gt_cache   = {}
        self.do_cache   = cache_gt

    def __len__(self):
        return len(self.pairs)

    def _get_gt(self, gt_path):
        if self.do_cache and gt_path in self.gt_cache:
            return self.gt_cache[gt_path]
        gt = load_gt_rgb(gt_path, scale=self.scale)
        if self.do_cache:
            self.gt_cache[gt_path] = gt
        return gt

    def _get_input(self, in_path, amp):
        return load_input_raw(in_path, amp_ratio=amp, scale=self.scale)

    @staticmethod
    def _augment(inp, gt):
        if random.random() > 0.5:
            inp = inp[:, ::-1, :].copy();  gt = gt[:, ::-1, :].copy()
        if random.random() > 0.5:
            inp = inp[::-1, :, :].copy();  gt = gt[::-1, :, :].copy()
        k = random.randint(0, 3)
        if k > 0:
            inp = np.rot90(inp, k, (0, 1)).copy()
            gt  = np.rot90(gt,  k, (0, 1)).copy()
        return inp, gt

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        amp  = float(pair["amp_ratio"])
        gt   = self._get_gt(pair["gt_path"])
        inp  = self._get_input(pair["input_path"], amp)

        if self.is_train:
            ps   = self.patch_size
            ps_h = ps // 2
            H_in, W_in = inp.shape[:2]
            if H_in < ps_h or W_in < ps_h:
                ps_h = min(H_in, W_in)
                ps   = ps_h * 2
            max_y = max(0, H_in - ps_h)
            max_x = max(0, W_in - ps_h)
            y_in  = random.randint(0, max_y)
            x_in  = random.randint(0, max_x)
            y_gt, x_gt = y_in * 2, x_in * 2
            inp = inp[y_in:y_in+ps_h, x_in:x_in+ps_h, :]
            gt  = gt[y_gt:y_gt+ps,   x_gt:x_gt+ps,   :]
            inp, gt = self._augment(inp, gt)

        inp_t = torch.from_numpy(np.transpose(inp, (2, 0, 1)).copy())
        gt_t  = torch.from_numpy(np.transpose(gt,  (2, 0, 1)).copy())
        return inp_t, gt_t


# -- Create datasets & loaders --
train_dataset = SIDDataset(train_pairs, patch_size=PATCH_SIZE,
                           scale=SCALE_FACTOR, is_train=True,  cache_gt=False)
val_dataset   = SIDDataset(val_pairs,   patch_size=PATCH_SIZE,
                           scale=SCALE_FACTOR, is_train=False, cache_gt=False)
test_dataset  = SIDDataset(test_pairs,  patch_size=PATCH_SIZE,
                           scale=SCALE_FACTOR, is_train=False, cache_gt=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_dataset,  batch_size=1, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} samples -> {len(train_loader)} batches")
print(f"Val  : {len(val_dataset)} samples")
print(f"Test : {len(test_dataset)} samples")

x, y = next(iter(train_loader))
print(f"Input  batch : {tuple(x.shape)}  dtype={x.dtype}  "
      f"range=[{x.min():.2f}, {x.max():.2f}]")
print(f"Target batch : {tuple(y.shape)}  dtype={y.dtype}  "
      f"range=[{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# -- DataLoader batch visualisation --
def packed_to_display(packed: torch.Tensor) -> np.ndarray:
    """4-channel packed Bayer -> 3-channel uint8 for display."""
    p = packed.cpu().float()
    rgb = torch.stack([p[0], (p[1]+p[3])/2, p[2]], dim=0)
    img = rgb.numpy().transpose(1, 2, 0)
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)


fig, axes = plt.subplots(2, BATCH_SIZE, figsize=(3 * BATCH_SIZE, 6))
fig.suptitle("DataLoader batch -- top: amplified raw input, bottom: GT",
             fontsize=11)
for i in range(BATCH_SIZE):
    axes[0, i].imshow(packed_to_display(x[i]))
    axes[0, i].axis("off")
    axes[1, i].imshow(y[i].numpy().transpose(1, 2, 0))
    axes[1, i].axis("off")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "dataloader_batch.png"), dpi=120, bbox_inches="tight")
plt.show()

---
## Section 8 -- U-Net Model

| Block | Channels | Notes |
|-------|----------|---------|
| Input | 4 | Packed Bayer |
| Enc 1-4 | 32->64->128->256 | 2x(Conv3x3 + LeakyReLU) + MaxPool |
| Bottleneck | 512 | 2x(Conv3x3 + LeakyReLU) |
| Dec 4-1 | 256->128->64->32 | ConvTranspose + skip + 2xConv |
| Output | 12 -> PixelShuffle(2) -> 3 | Full-resolution sRGB |

No batch normalisation (hurts on varying amp ratios). No residuals.

In [ ]:
# -- Model definition --

def conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch,  out_ch, 3, padding=1),
        nn.LeakyReLU(0.2, inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.LeakyReLU(0.2, inplace=True),
    )


class SIDUNet(nn.Module):
    """U-Net for raw low-light image processing (Sony subset).

    Input  : (B, 4, H/2, W/2)  -- packed amplified Bayer
    Output : (B, 3, H,   W)    -- predicted sRGB in [0,1]
    """

    def __init__(self, in_channels=4, base_ch=32):
        super().__init__()
        ch = base_ch
        self.enc1       = conv_block(in_channels, ch)
        self.pool1      = nn.MaxPool2d(2)
        self.enc2       = conv_block(ch,   ch*2)
        self.pool2      = nn.MaxPool2d(2)
        self.enc3       = conv_block(ch*2, ch*4)
        self.pool3      = nn.MaxPool2d(2)
        self.enc4       = conv_block(ch*4, ch*8)
        self.pool4      = nn.MaxPool2d(2)
        self.bottleneck = conv_block(ch*8, ch*16)
        self.up4  = nn.ConvTranspose2d(ch*16, ch*8,  2, stride=2)
        self.dec4 = conv_block(ch*16, ch*8)
        self.up3  = nn.ConvTranspose2d(ch*8,  ch*4,  2, stride=2)
        self.dec3 = conv_block(ch*8,  ch*4)
        self.up2  = nn.ConvTranspose2d(ch*4,  ch*2,  2, stride=2)
        self.dec2 = conv_block(ch*4,  ch*2)
        self.up1  = nn.ConvTranspose2d(ch*2,  ch,    2, stride=2)
        self.dec1 = conv_block(ch*2,  ch)
        self.out_conv      = nn.Conv2d(ch, 12, 1)
        self.pixel_shuffle = nn.PixelShuffle(2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, a=0.2,
                                        nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b  = self.bottleneck(self.pool4(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.clamp(self.pixel_shuffle(self.out_conv(d1)), 0.0, 1.0)


# -- Instantiate with multi-GPU support --
_model = SIDUNet(in_channels=4, base_ch=32)

if n_gpus > 1:
    print(f"Using DataParallel across {n_gpus} GPUs")
    model = nn.DataParallel(_model)
else:
    model = _model

model = model.to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters -- total: {total_p:,}  trainable: {trainable_p:,}")

with torch.no_grad():
    _in  = torch.rand(1, 4, PATCH_SIZE//2, PATCH_SIZE//2, device=device)
    _out = model(_in)
print(f"Smoke test  input : {tuple(_in.shape)}")
print(f"Smoke test  output: {tuple(_out.shape)}")
assert _out.shape == (1, 3, PATCH_SIZE, PATCH_SIZE)
print("Model architecture OK.")

In [ ]:
# -- Parameter breakdown by layer group --
layer_groups = {
    "Encoder":    ["enc1", "enc2", "enc3", "enc4"],
    "Bottleneck": ["bottleneck"],
    "Decoder":    ["dec1", "dec2", "dec3", "dec4", "up1", "up2", "up3", "up4"],
    "Output":     ["out_conv", "pixel_shuffle"],
}

base = _model
group_params = {}
for grp, layer_names in layer_groups.items():
    n = sum(
        sum(p.numel() for p in getattr(base, ln).parameters())
        for ln in layer_names if hasattr(base, ln)
    )
    group_params[grp] = n

fig, ax = plt.subplots(figsize=(8, 4))
grps   = list(group_params.keys())
vals   = [group_params[g] / 1e6 for g in grps]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
bars   = ax.bar(grps, vals, color=colors, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, labels=[f"{v:.2f}M" for v in vals], padding=3)
ax.set_ylabel("Parameters (millions)")
ax.set_title("SIDUNet -- Parameter Count by Layer Group")
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "model_params.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Total: {sum(vals):.2f}M parameters")

---
## Section 9 -- Training

| Detail | Value |
|--------|-------|
| Loss | L1 |
| Optimiser | Adam (b1=0.9, b2=0.999) |
| LR | 1e-4 -> 1e-5 at epoch 250 |
| Epochs | 500 |
| Crop | 256x256 |
| Augmentation | Random flip + 90 deg rotation |
| Precision | AMP mixed-precision |
| Multi-GPU | DataParallel (T4 x 2) |

In [ ]:
# -- Training utilities --
criterion   = nn.L1Loss()
optimizer   = torch.optim.Adam(
    model.parameters(), lr=LR_INITIAL, betas=(0.9, 0.999), eps=1e-8)
_amp_device = "cuda" if torch.cuda.is_available() else "cpu"
scaler      = GradScaler(_amp_device, enabled=torch.cuda.is_available())


def set_lr(opt, lr: float):
    for g in opt.param_groups:
        g["lr"] = lr


def train_epoch(model, loader, optimizer, scaler, device) -> float:
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        inputs  = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            preds = model(inputs)
            loss  = criterion(preds, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device) -> dict:
    model.eval()
    psnrs, ssims, losses = [], [], []
    for inputs, targets in loader:
        inputs  = inputs.to(device)
        targets = targets.to(device)
        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            preds = model(inputs)
            loss  = criterion(preds, targets)
        losses.append(loss.item())
        pred_np = np.clip(
            preds[0].cpu().float().numpy().transpose(1, 2, 0), 0, 1)
        tgt_np  = np.clip(
            targets[0].cpu().float().numpy().transpose(1, 2, 0), 0, 1)
        p = psnr_fn(tgt_np, pred_np, data_range=1.0)
        _md = min(tgt_np.shape[:2])
        ws  = max(3, min(7, _md if _md % 2 == 1 else _md - 1))
        s   = ssim_fn(tgt_np, pred_np, data_range=1.0,
                      channel_axis=2, win_size=ws)
        psnrs.append(p);  ssims.append(s)
    return {"loss": float(np.mean(losses)),
            "psnr": float(np.mean(psnrs)),
            "ssim": float(np.mean(ssims))}


def save_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        "epoch": epoch,
        "model_state": (model.module.state_dict()
                        if hasattr(model, "module")
                        else model.state_dict()),
        "optimizer_state": optimizer.state_dict(),
        "metrics": metrics
    }, path)


def load_checkpoint(model, optimizer, path) -> int:
    ckpt  = torch.load(path, map_location=device)
    state = ckpt["model_state"]
    if hasattr(model, "module"):
        model.module.load_state_dict(state)
    else:
        model.load_state_dict(state)
    optimizer.load_state_dict(ckpt["optimizer_state"])
    ep = ckpt["epoch"]
    psnr_str = ckpt["metrics"].get("psnr", "?")
    print(f"Resumed from epoch {ep}  PSNR={psnr_str}")
    return ep


print("Training utilities ready.")

In [ ]:
# -- Learning-rate schedule visualisation --
ep_range = np.arange(1, NUM_EPOCHS + 1)
lr_sched = np.where(ep_range <= LR_DECAY_EP, LR_INITIAL, LR_DECAYED)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ep_range, lr_sched, color="#4C72B0", linewidth=2)
ax.axvline(LR_DECAY_EP, color="red", linestyle="--",
           label=f"Decay at epoch {LR_DECAY_EP}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(str(RESULT_DIR / "lr_schedule.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- Resume from checkpoint (optional) --
START_EPOCH = 0
BEST_PSNR   = 0.0
RESUME_CKPT = None  # set to a checkpoint path to resume

if RESUME_CKPT and Path(RESUME_CKPT).exists():
    START_EPOCH = load_checkpoint(model, optimizer, RESUME_CKPT)
else:
    print("Starting training from scratch.")

history = {
    "train_loss": [], "val_loss": [], "val_psnr": [],
    "val_ssim": [], "val_epochs": []
}

In [ ]:
# =========================================================================
#  MAIN TRAINING LOOP
# =========================================================================

print(f"Training {NUM_EPOCHS} epochs | device={device} | "
      f"n_gpus={n_gpus} | batch={BATCH_SIZE}")
print("=" * 72)

t0 = time.time()

for epoch in range(START_EPOCH + 1, NUM_EPOCHS + 1):

    current_lr = LR_INITIAL if epoch <= LR_DECAY_EP else LR_DECAYED
    set_lr(optimizer, current_lr)

    train_loss = train_epoch(model, train_loader, optimizer, scaler, device)
    history["train_loss"].append(train_loss)

    if epoch % VAL_EVERY == 0 or epoch == 1:
        val_metrics = evaluate(model, val_loader, device)
        history["val_loss"].append(val_metrics["loss"])
        history["val_psnr"].append(val_metrics["psnr"])
        history["val_ssim"].append(val_metrics["ssim"])
        history["val_epochs"].append(epoch)

        elapsed = time.time() - t0
        print(f"Ep {epoch:4d}/{NUM_EPOCHS} | LR={current_lr:.0e} | "
              f"TrainL1={train_loss:.4f} | ValL1={val_metrics["loss"]:.4f} | "
              f"PSNR={val_metrics["psnr"]:.2f} dB | "
              f"SSIM={val_metrics["ssim"]:.4f} | "
              f"t={elapsed/60:.1f}min")

        if val_metrics["psnr"] > BEST_PSNR:
            BEST_PSNR = val_metrics["psnr"]
            save_checkpoint(model, optimizer, epoch, val_metrics,
                            str(CKPT_DIR / "best_model.pth"))
            print(f"  New best PSNR: {BEST_PSNR:.2f} dB")

    if epoch % SAVE_EVERY == 0:
        save_checkpoint(model, optimizer, epoch, {"psnr": BEST_PSNR},
                        str(CKPT_DIR / f"epoch_{epoch:04d}.pth"))

print(f"\nTraining complete.  Best val PSNR: {BEST_PSNR:.2f} dB")

---
## Section 10 -- Training Curves

In [ ]:
# -- Training curves --
def plot_training_history(history):
    ep_train = list(range(1, len(history["train_loss"]) + 1))
    ep_val   = history["val_epochs"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Training History", fontsize=14, fontweight="bold")

    axes[0].plot(ep_train, history["train_loss"],
                 label="Train L1", color="steelblue", linewidth=1.5)
    if history["val_loss"]:
        axes[0].plot(ep_val, history["val_loss"],
                     label="Val L1", color="tomato", linewidth=1.5)
    axes[0].set_title("L1 Loss"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("L1 Loss"); axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if history["val_psnr"]:
        axes[1].plot(ep_val, history["val_psnr"],
                     color="green", linewidth=1.5, marker="o", markersize=4)
        axes[1].axhline(max(history["val_psnr"]), color="green",
                        linestyle="--", alpha=0.5,
                        label=f"Best={max(history['val_psnr']):.2f} dB")
    axes[1].set_title("Validation PSNR (dB)"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("PSNR (dB)"); axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    if history["val_ssim"]:
        axes[2].plot(ep_val, history["val_ssim"],
                     color="purple", linewidth=1.5, marker="o", markersize=4)
        axes[2].axhline(max(history["val_ssim"]), color="purple",
                        linestyle="--", alpha=0.5,
                        label=f"Best={max(history['val_ssim']):.4f}")
    axes[2].set_title("Validation SSIM"); axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("SSIM"); axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(RESULT_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {RESULT_DIR}/training_curves.png")


if any(v for v in history.values() if isinstance(v, list) and v):
    plot_training_history(history)
else:
    print("No training history yet.")

In [ ]:
# -- Smoothed loss on log scale --
if len(history["train_loss"]) > 5:
    losses   = np.array(history["train_loss"])
    window   = max(1, len(losses) // 20)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, len(losses)+1), losses,
            alpha=0.3, color="steelblue", label="Raw")
    ax.plot(range(window, len(losses)+1), smoothed,
            color="steelblue", linewidth=2,
            label=f"Smoothed (window={window})")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Train L1 Loss (log scale)")
    ax.set_title("Training Loss (log scale)")
    ax.legend(); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULT_DIR / "training_loss_log.png"), dpi=150, bbox_inches="tight")
    plt.show()

---
## Section 11 -- Evaluation

In [ ]:
# -- Load best checkpoint --
best_ckpt = CKPT_DIR / "best_model.pth"
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=device)
    if hasattr(model, "module"):
        model.module.load_state_dict(ckpt["model_state"])
    else:
        model.load_state_dict(ckpt["model_state"])
    ep_str   = ckpt["epoch"]
    psnr_str = ckpt["metrics"].get("psnr", "?")
    print(f"Loaded best model -- epoch {ep_str}  PSNR={psnr_str}")
else:
    print("No checkpoint found -- using current model weights.")

In [ ]:
# -- Test-set evaluation --
test_metrics = evaluate(model, test_loader, device)
print("\n-- Test-set results --")
print(f"  L1 Loss : {test_metrics['loss']:.4f}")
print(f"  PSNR    : {test_metrics['psnr']:.2f} dB")
print(f"  SSIM    : {test_metrics['ssim']:.4f}")
print("--")
print("Paper baseline (full-res, 4000 ep): ~28.88 dB / 0.787 SSIM")
print("Lower values expected (scaled images, fewer epochs, 20% data).")

In [ ]:
# -- Per-sample metrics --
model.eval()
per_sample = []

with torch.no_grad():
    for idx, (inputs, targets) in enumerate(
            tqdm(test_loader, desc="Per-sample eval")):
        inputs  = inputs.to(device)
        targets = targets.to(device)
        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            preds = model(inputs)
        pred_np = np.clip(
            preds[0].cpu().float().numpy().transpose(1,2,0), 0, 1)
        tgt_np  = np.clip(
            targets[0].cpu().float().numpy().transpose(1,2,0), 0, 1)
        p = psnr_fn(tgt_np, pred_np, data_range=1.0)
        _md = min(tgt_np.shape[:2])
        ws  = max(3, min(7, _md if _md % 2 == 1 else _md - 1))
        s   = ssim_fn(tgt_np, pred_np, data_range=1.0,
                      channel_axis=2, win_size=ws)
        pair = test_pairs[idx] if idx < len(test_pairs) else {}
        per_sample.append({
            "idx": idx, "psnr": p, "ssim": s,
            "amp": pair.get("amp_ratio", 0)
        })

psnrs_ps = [r["psnr"] for r in per_sample]
ssims_ps = [r["ssim"] for r in per_sample]
amps_ps  = [r["amp"]  for r in per_sample]

print(f"PSNR  min={min(psnrs_ps):.2f}  max={max(psnrs_ps):.2f}  "
      f"mean={np.mean(psnrs_ps):.2f} dB")
print(f"SSIM  min={min(ssims_ps):.4f}  max={max(ssims_ps):.4f}  "
      f"mean={np.mean(ssims_ps):.4f}")

In [ ]:
# -- Metrics visualisation --
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Test-Set Metric Analysis", fontsize=14, fontweight="bold")

# PSNR histogram
axes[0,0].hist(psnrs_ps, bins=20, color="green", edgecolor="white", alpha=0.8)
axes[0,0].axvline(np.mean(psnrs_ps), color="red", linestyle="--",
                  label=f"Mean={np.mean(psnrs_ps):.2f} dB")
axes[0,0].set_xlabel("PSNR (dB)"); axes[0,0].set_ylabel("Count")
axes[0,0].set_title("PSNR Distribution"); axes[0,0].legend()

# SSIM histogram
axes[0,1].hist(ssims_ps, bins=20, color="purple", edgecolor="white", alpha=0.8)
axes[0,1].axvline(np.mean(ssims_ps), color="red", linestyle="--",
                  label=f"Mean={np.mean(ssims_ps):.4f}")
axes[0,1].set_xlabel("SSIM"); axes[0,1].set_ylabel("Count")
axes[0,1].set_title("SSIM Distribution"); axes[0,1].legend()

# PSNR vs amp
axes[1,0].scatter(amps_ps, psnrs_ps, alpha=0.6, c="green", s=50)
axes[1,0].set_xlabel("Amplification Ratio"); axes[1,0].set_ylabel("PSNR (dB)")
axes[1,0].set_title("PSNR vs Amplification Ratio")

# SSIM vs amp
axes[1,1].scatter(amps_ps, ssims_ps, alpha=0.6, c="purple", s=50)
axes[1,1].set_xlabel("Amplification Ratio"); axes[1,1].set_ylabel("SSIM")
axes[1,1].set_title("SSIM vs Amplification Ratio")

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "test_metric_analysis.png"), dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# -- Visual comparison: Input | Prediction | GT --
def tensor_to_image(t: torch.Tensor) -> np.ndarray:
    img = t.cpu().float().numpy().transpose(1, 2, 0)
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)


@torch.no_grad()
def visualise_samples(model, dataset, n_samples=4, save_dir=RESULT_DIR):
    model.eval()
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    fig, axes = plt.subplots(n_samples, 3, figsize=(14, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle("Visual Comparison -- Input | Prediction | Ground Truth",
                 fontsize=13, fontweight="bold")
    for j, t in enumerate(["Input (amplified raw)", "Network Output",
                            "Ground Truth"]):
        axes[0,j].set_title(t, fontsize=10, fontweight="bold")

    for row, idx in enumerate(indices):
        inp_t, gt_t = dataset[idx]
        inp_t = inp_t.unsqueeze(0).to(device)
        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            pred_t = model(inp_t)[0]
        inp_disp  = packed_to_display(inp_t[0])
        pred_disp = tensor_to_image(pred_t)
        gt_disp   = tensor_to_image(gt_t)

        p_val = psnr_fn(gt_disp/255.0, pred_disp/255.0, data_range=1.0)
        _md   = min(gt_disp.shape[:2])
        ws    = max(3, min(7, _md if _md % 2 == 1 else _md - 1))
        s_val = ssim_fn(gt_disp/255.0, pred_disp/255.0,
                        data_range=1.0, channel_axis=2, win_size=ws)

        for j, img in enumerate([inp_disp, pred_disp, gt_disp]):
            axes[row, j].imshow(img); axes[row, j].axis("off")
        axes[row, 1].set_title(
            f"PSNR={p_val:.1f} dB  SSIM={s_val:.3f}",
            fontsize=9, color="darkgreen")

        imageio.imwrite(str(save_dir / f"pred_{idx:04d}.png"), pred_disp)
        imageio.imwrite(str(save_dir / f"gt_{idx:04d}.png"),   gt_disp)

    plt.tight_layout()
    plt.savefig(str(save_dir / "visual_comparison.png"), dpi=120,
                bbox_inches="tight")
    plt.show()


visualise_samples(model, test_dataset, n_samples=min(4, len(test_dataset)))

In [ ]:
# -- Error maps --
@torch.no_grad()
def plot_error_maps(model, dataset, n_samples=3, save_dir=RESULT_DIR):
    model.eval()
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle("Error Maps  |Prediction - GT|",
                 fontsize=13, fontweight="bold")
    for j, h in enumerate(["Prediction", "Ground Truth",
                            "Error Map", "Error Histogram"]):
        axes[0,j].set_title(h, fontsize=10, fontweight="bold")

    for row, idx in enumerate(indices):
        inp_t, gt_t = dataset[idx]
        inp_t = inp_t.unsqueeze(0).to(device)
        with autocast(_amp_device, enabled=torch.cuda.is_available()):
            pred_t = model(inp_t)[0]
        pred_np = np.clip(
            pred_t.cpu().float().numpy().transpose(1,2,0), 0, 1)
        gt_np   = np.clip(
            gt_t.cpu().float().numpy().transpose(1,2,0),   0, 1)
        err_map = np.abs(pred_np - gt_np).mean(axis=2)

        axes[row,0].imshow(pred_np); axes[row,0].axis("off")
        axes[row,1].imshow(gt_np);   axes[row,1].axis("off")
        im = axes[row,2].imshow(err_map, cmap="hot", vmin=0, vmax=0.3)
        plt.colorbar(im, ax=axes[row,2], shrink=0.8)
        axes[row,2].axis("off")
        axes[row,3].hist(err_map.flatten(), bins=50,
                         color="orangered", edgecolor="white", alpha=0.8)
        axes[row,3].set_xlabel("|Error|"); axes[row,3].set_ylabel("Count")
        axes[row,3].set_title(f"MAE={err_map.mean():.4f}", fontsize=9)

    plt.tight_layout()
    plt.savefig(str(save_dir / "error_maps.png"), dpi=120, bbox_inches="tight")
    plt.show()


plot_error_maps(model, test_dataset, n_samples=min(3, len(test_dataset)))

---
## Section 12 -- Performance Summary

In [ ]:
# -- Performance table --
paper_results = {
    "U-Net (paper, full-res, 4000ep)":  (28.88, 0.787),
    "CAN (paper)":                       (27.40, 0.792),
    "Raw->sRGB input (paper)":           (17.40, 0.554),
    "L1->SSIM loss (paper)":             (28.64, 0.817),
    "L1->L2 loss (paper)":               (28.47, 0.784),
    "Packed->Masked Bayer (paper)":      (26.95, 0.744),
}

our_psnr = test_metrics["psnr"]
our_ssim = test_metrics["ssim"]

print(f"{"Method":<55}  {"PSNR dB":>8}  {"SSIM":>8}")
print("-" * 75)
label_ours = f"Ours (20% data, scale={SCALE_FACTOR}, {NUM_EPOCHS}ep)"
print(f"{label_ours:<55}  {our_psnr:>8.2f}  {our_ssim:>8.4f}")
print("-" * 75)
for name, (p, s) in paper_results.items():
    print(f"{name:<55}  {p:>8.2f}  {s:>8.4f}")

In [ ]:
# -- Bar chart: ours vs. paper --
methods    = ["Ours"] + list(paper_results.keys())
psnr_vals  = [our_psnr] + [v[0] for v in paper_results.values()]
ssim_vals  = [our_ssim] + [v[1] for v in paper_results.values()]
bar_colors = ["#C44E52"] + ["#4C72B0"] * len(paper_results)

short_names = [
    "Ours\n(this run)", "U-Net\n(paper)", "CAN\n(paper)",
    "sRGB\ninput", "SSIM\nloss", "L2\nloss", "Masked\nBayer"
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Performance Comparison: Ours vs. Paper",
             fontsize=13, fontweight="bold")

b0 = axes[0].bar(short_names, psnr_vals, color=bar_colors, edgecolor="white")
axes[0].bar_label(b0, labels=[f"{v:.1f}" for v in psnr_vals], padding=3)
axes[0].set_ylabel("PSNR (dB)"); axes[0].set_title("PSNR Comparison")
axes[0].set_ylim(0, max(psnr_vals) + 3)

b1 = axes[1].bar(short_names, ssim_vals, color=bar_colors, edgecolor="white")
axes[1].bar_label(b1, labels=[f"{v:.3f}" for v in ssim_vals], padding=3)
axes[1].set_ylabel("SSIM"); axes[1].set_title("SSIM Comparison")
axes[1].set_ylim(0, max(ssim_vals) + 0.1)

ours_p  = mpatches.Patch(color="#C44E52", label="Ours")
paper_p = mpatches.Patch(color="#4C72B0", label="Paper")
fig.legend(handles=[ours_p, paper_p], loc="upper right")

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "performance_comparison.png"), dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# -- Resource summary --
total_train_time = time.time() - t0
epochs_done  = len(history["train_loss"])
time_per_ep  = total_train_time / epochs_done if epochs_done > 0 else 0

print("=" * 55)
print("  TRAINING RESOURCE SUMMARY")
print("=" * 55)
print(f"  Epochs trained   : {epochs_done}")
print(f"  Total time       : {total_train_time/60:.1f} min")
print(f"  Time per epoch   : {time_per_ep:.1f} s")
print(f"  Scale factor     : {SCALE_FACTOR}")
print(f"  Dataset fraction : {SAMPLE_FRAC*100:.0f}%")
print(f"  Train pairs      : {len(train_pairs)}")
print(f"  Val pairs        : {len(val_pairs)}")
print(f"  Test pairs       : {len(test_pairs)}")
print(f"  Best val PSNR    : {BEST_PSNR:.2f} dB")
print(f"  Test PSNR        : {test_metrics['psnr']:.2f} dB")
print("=" * 55)

---
## Section 13 -- Inference on a Single Raw Image

Use this cell to run the trained model on any new `.ARW` file.

In [ ]:
# -- Single-image inference --

@torch.no_grad()
def infer_single(raw_path, amp_ratio, model, device,
                 scale=SCALE_FACTOR, save_path=None):
    """Run trained model on one .ARW file.
    raw_path  -- path to input raw file
    amp_ratio -- exposure amplification (e.g. 100, 250, 300)
    Returns predicted sRGB uint8 array (H, W, 3).
    """
    model.eval()
    packed = load_input_raw(raw_path, amp_ratio=amp_ratio, scale=scale)
    inp_t  = torch.from_numpy(
        np.transpose(packed, (2, 0, 1))[np.newaxis]
    ).to(device)
    with autocast(_amp_device, enabled=torch.cuda.is_available()):
        pred = model(inp_t)[0]
    out = (np.clip(
        pred.cpu().float().numpy().transpose(1, 2, 0), 0, 1) * 255
    ).astype(np.uint8)
    if save_path:
        imageio.imwrite(save_path, out)
        print(f"Saved -> {save_path}")
    return out


# -- Demo: use the first test image --
if test_pairs:
    eg = test_pairs[0]
    pred_img = infer_single(
        eg["input_path"], eg["amp_ratio"], model, device,
        save_path=str(RESULT_DIR / "inference_output.png")
    )
    raw_packed = load_input_raw(
        eg["input_path"], eg["amp_ratio"], SCALE_FACTOR)
    inp_tensor = torch.from_numpy(
        np.transpose(raw_packed, (2, 0, 1)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Inference Demo -- amp x{eg["amp_ratio"]:.0f}",
                 fontsize=12)
    axes[0].imshow(packed_to_display(inp_tensor))
    axes[0].set_title("Input (amplified raw)"); axes[0].axis("off")
    axes[1].imshow(pred_img)
    axes[1].set_title("Network output"); axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(str(RESULT_DIR / "inference_demo.png"), dpi=120,
                bbox_inches="tight")
    plt.show()
else:
    print("No test pairs available.")

---
## Summary

This notebook implements the full **Learning to See in the Dark** pipeline for Kaggle GPU T4 x2.

| Component | Detail |
|-----------|--------|
| Dataset | SID Sony 2025 -- 20% of scenes, scale 0.25 |
| Dataset source | Kaggle input: `marcorosato/sid-dataset` (no download needed) |
| Preprocessing | Bayer packing (4ch), black-level, amplification 100-300x |
| Model | U-Net (5-level) + PixelShuffle output, ~7.7M params |
| Multi-GPU | `nn.DataParallel` across both T4 GPUs |
| Training | L1 loss, Adam, AMP, step LR, 500 epochs |
| Metrics | PSNR / SSIM per sample and aggregate |

### Saved Artefacts (`/kaggle/working/results/`)
- `eda_overview.png`, `eda_amp_buckets.png`, `eda_sample_images.png`
- `eda_channel_stats.png`, `eda_channel_corr.png`, `eda_noise_vs_amp.png`
- `preprocessing_vis.png`, `dataloader_batch.png`
- `model_params.png`, `lr_schedule.png`
- `training_curves.png`, `training_loss_log.png`
- `test_metric_analysis.png`, `visual_comparison.png`, `error_maps.png`
- `performance_comparison.png`, `inference_output.png`

### To improve performance
- Increase `SAMPLE_FRAC` (e.g. 0.50 or 1.0)
- Increase `SCALE_FACTOR` (e.g. 0.5)
- Increase `NUM_EPOCHS` (e.g. 2000 or 4000)
- Increase `PATCH_SIZE` (e.g. 512)